# Calculating Across Rows

How do we calculate row-wise in Pandas like we do in Excel?

<img src="https://sandeepmj.github.io/image-host/oil-v-gold.png">


We need a couple of new concepts to do this calculation:

1. Capture a number from a cell and hold in variable.
2. We `apply()`calculations across columns, row-wise, using using a `lambda expression`.


This is step one in your ability to accomplish tasks that are highly specialized to your beats.



In [1]:
## import library
import pandas as pd

In [2]:
## pull in data
df = pd.read_csv("https://raw.githubusercontent.com/sandeepmj/datasets/refs/heads/main/oil-v-gold-pandas.csv")
df

,Date,PX_GOLD,PX_OIL
0,1/31/08,922.7,91.75
1,2/29/08,975.0,101.84
2,3/31/08,916.2,101.58
3,4/30/08,865.1,113.46
4,5/30/08,887.3,127.35
...,...,...,...
116,9/29/17,1281.5,51.67
117,10/31/17,1270.5,54.38
118,11/30/17,1273.2,57.40
119,12/29/17,1309.3,60.42


What are the different values for row **TWO** in this formula for `PX_GOLD`:

$$\text{Pct Chg at time } t = \frac{\text{Value at time } t - \text{Value at start time}}{\text{Value at start time}} \times 100$$

Use `.iat` to store a value from a specific cell into a variable:

```dataframe["column_name"].iat[row_index_number]```

In [10]:
## hold gold
gold_base = df["PX_GOLD"].iat[0]
gold_base

np.float64(922.7)

# Two types of `lambda expressions`



In [11]:
## method 1
df["gold_index"] = df["PX_GOLD"].apply(lambda x: ((x - gold_base) / gold_base) * 100)
df

,Date,PX_GOLD,PX_OIL,gold_index
0,1/31/08,922.7,91.75,0.000000
1,2/29/08,975.0,101.84,5.668148
2,3/31/08,916.2,101.58,-0.704454
3,4/30/08,865.1,113.46,-6.242549
4,5/30/08,887.3,127.35,-3.836567
...,...,...,...,...
116,9/29/17,1281.5,51.67,38.885878
117,10/31/17,1270.5,54.38,37.693725
118,11/30/17,1273.2,57.40,37.986344
119,12/29/17,1309.3,60.42,41.898775


In [14]:
## method 2
df["gold_index2"] = df.apply(lambda x: (x["PX_GOLD"]- gold_base) / gold_base * 100, axis = 1 )
df

,Date,PX_GOLD,PX_OIL,gold_index,gold_index2
0,1/31/08,922.7,91.75,0.000000,0.000000
1,2/29/08,975.0,101.84,5.668148,5.668148
2,3/31/08,916.2,101.58,-0.704454,-0.704454
3,4/30/08,865.1,113.46,-6.242549,-6.242549
4,5/30/08,887.3,127.35,-3.836567,-3.836567
...,...,...,...,...,...
116,9/29/17,1281.5,51.67,38.885878,38.885878
117,10/31/17,1270.5,54.38,37.693725,37.693725
118,11/30/17,1273.2,57.40,37.986344,37.986344
119,12/29/17,1309.3,60.42,41.898775,41.898775


Now compare **method 1**:

```python
df["PX_GOLD"].apply(lambda x: ((x - gold_baseline) / gold_baseline) * 100)
```

to **method 2**:

```python
df.apply(lambda x: (x["PX_GOLD"]- gold_baseline) / gold_baseline * 100, axis = 1 )
```

What are the differences?

## Method 1: What  `x` stand for:

- In Pandas, we `apply()` lambda expressions to columns.
-  `.apply()`  works element-wise on each value in the `PX_GOLD` column, one at a time.
- `x` is a single value representing a cell in the column `PX_GOLD`
- The ```lambda x:``` processes each row of a dataframe, one row at a time.
-  The `lambda expression` takes each individual number `x` from `PX_GOLD` and runs the indexing formula on it.

Think of it as: "call each value in `PX_GOLD` as `x`, and do this math to using `x` in the formula." 

## Method 2: What `x` stands for:

Will get to `axis = 1` in a moment, but for now:

In method 1, `df["PX_GOLD"].apply(lambda x:...)`, the `lambda x` holds just the value of `df["PX_GOLD"]` for that row, one row at a time.

In method 2,  `df.apply(lambda x:...)`, the `lambda x` holds the values of every item in every cell in that row, one row at a time.

So in method 2, the `lambda x` represents a single row (not the entire DataFrame, not a single cell, but the entire row.

Then, `x['PX_GOLD']` in the expression extracts the value of `PX_GOLD` from the current row, one row at a time.

## So what is ```axis = 1``` in Pandas?

`df["gold_index2"] = df.apply(lambda x: (x["PX_GOLD"]- gold_baseline) / gold_baseline * 100, axis = 1 )`

Basically, the axis parameter determines whether the operation is applied to rows or columns:

<img src="https://raw.githubusercontent.com/sandeepmj/22spring-advanced-data/main/img/axis.png" width="75%">

**Confession**:

I almost always forget to add `axis = 1` to my lambda expressions. 

Look for the error `KeyError: 'some_column'` where it references the column you are trying to access. 

The moment I see that error, I know I need to add `axis = 1` to make it work.

# Apply Lambda across multiple columns

In [15]:
## create a df with mock data

data = {'firm': ['Bilk Inc.', 'Vine & Co.', 'Kiln Inc.', 'Y & Y Consulting', 'Trending Inc.', "State Insurance"],
   'net_2024': [1150, 2300, 3400, 1500, 6500, 1000 ],
        'net_2025': [1216, 2619, 3701, 1890, 5630, 1099 ]
           }
df = pd.DataFrame(data)

df

,firm,net_2024,net_2025
0,Bilk Inc.,1150,1216
1,Vine & Co.,2300,2619
2,Kiln Inc.,3400,3701
3,Y & Y Consulting,1500,1890
4,Trending Inc.,6500,5630
5,State Insurance,1000,1099


In [24]:
## calculate using lambda
df["pct_chg"] = df.apply(lambda x: (x["net_2025"]-x["net_2024"])/x["net_2024"] * 100, axis = 1)
df

,firm,net_2024,net_2025,pct_chg
0,Bilk Inc.,1150,1216,5.739130
1,Vine & Co.,2300,2619,13.869565
2,Kiln Inc.,3400,3701,8.852941
3,Y & Y Consulting,1500,1890,26.000000
4,Trending Inc.,6500,5630,-13.384615
5,State Insurance,1000,1099,9.900000


We want to calculate the percent change between 2024 and 2025

$$\text{Percent Change} = \frac{\text{new value} - \text{old value}}{\text{old value}} \times 100$$

## Tenary Expression
```value1 if some_condition else value2```



In [29]:
## write our alcohol conditional as a tenary
age = 18
"legal drink available" if age >= 21 else "No drinks for you!"


'No drinks for you!'

In [ ]:
## change the age



In [ ]:
## call df again



In [36]:

## Categorize the difference between net2024 and net_2025 as increase or decrease

df["growth"] = df.apply(lambda x: "False" if x["net_2024"] > x["net_2025"] else "True", axis = 1)
df

,firm,net_2024,net_2025,pct_chg,growth
0,Bilk Inc.,1150,1216,5.739130,True
1,Vine & Co.,2300,2619,13.869565,True
2,Kiln Inc.,3400,3701,8.852941,True
3,Y & Y Consulting,1500,1890,26.000000,True
4,Trending Inc.,6500,5630,-13.384615,False
5,State Insurance,1000,1099,9.900000,True
